In [ ]:
import matplotlib.colors as mcolors
import matplotlib.transforms as mtransforms

def plot_field_columns(fig, ax, fields, fields_id, iteration,
                       Xmesh, Ymesh, Xgrid, Ygrid,
                       func, field_names, colormap, make_formatter):
    """
    Plot exact, computed, and residual fields (with colorbars) for each field.

    Parameters:
      - fig: matplotlib figure
      - ax: 2D array of axes with shape (3, 1+n_fields)
      - fields: list (or dict) of field data
      - fields_id: list of field ids to plot
      - iteration: current iteration index
      - Xmesh, Ymesh: meshgrid arrays for plotting (edges, n+1 x n+1)
      - Xgrid, Ygrid: meshgrid arrays for evaluation (centers, n x n)
      - func: function returning the exact field (used to compute field differences)
      - field_names: list of field names
      - colormap: colormap to use
      - make_formatter: function to create scalar formatter for colorbars

    Returns:
      (ims_ref, ims_field, ims_res)
    """
    ims_ref, ims_field, ims_res = [], [], []

    # Flatten evaluation grid for model calls
    eval_points = np.hstack((Xgrid.reshape(-1, 1), Ygrid.reshape(-1, 1)))
    fields_exact_all = func(eval_points).reshape(Xgrid.shape + (-1,))

    for i, field_id in enumerate(fields_id):
        # --- Exact solution (top row) ---
        ax_exact = ax[0][i]
        field_exact = fields_exact_all[..., field_id]
        field_norm = mcolors.Normalize(vmin=field_exact.min(), vmax=field_exact.max())
        im_ref = pcolor_plot(ax_exact, Xmesh, Ymesh, field_exact, field_names[field_id] + "*",
                             colormap=colormap, norm=field_norm)
        ims_ref.append(im_ref)

        # Colorbar for exact
        pos = ax_exact.get_position()
        cax_pos = mtransforms.Bbox.from_bounds(pos.x0 + pos.width*0.05, pos.y1 + 0.04,
                                               pos.width*0.9, 0.01)
        cax = fig.add_axes(cax_pos)
        cbfield = fig.colorbar(im_ref, cax=cax, orientation='horizontal', format=make_formatter())
        cbfield.ax.xaxis.set_ticks_position('top')

        # --- Predicted field (middle row) ---
        ax_field = ax[1][i]
        field_pred = np.array(fields[field_id][iteration]).reshape(Xgrid.shape)
        im_field = pcolor_plot(ax_field, Xmesh, Ymesh, field_pred, field_names[field_id],
                               colormap=colormap, norm=field_norm)
        ims_field.append(im_field)

        # --- Residual (bottom row) ---
        ax_res = ax[2][i]
        field_diff = field_pred - field_exact
        diff_norm = mcolors.Normalize(vmin=-np.abs(field_diff).max(), vmax=np.abs(field_diff).max())
        im_res = pcolor_plot(ax_res, Xmesh, Ymesh, field_diff, f"{field_names[field_id]} - {field_names[field_id]}*",
                             colormap='coolwarm', norm=diff_norm)
        ims_res.append(im_res)

        pos = ax_res.get_position()
        cax_pos = mtransforms.Bbox.from_bounds(pos.x0 + pos.width*0.05, pos.y0 - 0.03,
                                               pos.width*0.9, 0.01)
        cax = fig.add_axes(cax_pos)
        fig.colorbar(im_res, cax=cax, orientation='horizontal', format=make_formatter())

    return ims_ref, ims_field, ims_res


In [ ]:
# Field Plotting
from scipy.interpolate import RegularGridInterpolator
from matplotlib.patches import Rectangle


# --- Font size for printing ---
plotting_factor = 3
font_factor = 3
title_font_size = 6
axes_font_size = 5
plt.rcParams.update({
    "font.size": title_font_size*font_factor,         # Default font size for all text
    "figure.titlesize": title_font_size*font_factor, 
    "axes.labelsize": axes_font_size*font_factor,
    "xtick.labelsize": axes_font_size*font_factor,
    "ytick.labelsize": axes_font_size*font_factor,
    "legend.fontsize": axes_font_size*font_factor,
})

plot_metrics = 1 # 1 to plot metrics, 0 to not plot metrics
fields_plotted = [0, 1, 2, 3, 4, 5, 6, 7][:2] + [0, 1, 2, 3, 4, 5, 6, 7][5:]  # Select all fields except [2, 3, 4]
n_fields = len(fields_plotted)
field_names_tex = [r"$u_x$", r"$u_y$", r"$\varepsilon_{xx}$", r"$\varepsilon_{yy}$", r"$\varepsilon_{xy}$", r"$\sigma_{xx}$", r"$\sigma_{yy}$", r"$\sigma_{xy}$"]

# --- Custom colormap ---
cmap = plt.get_cmap('viridis')
num_colors = 20
color_values = [cmap(i) for i in np.linspace(0, 1, num_colors)]
cmap = mcolors.ListedColormap(color_values)

# --- Load FEM reference solution and create interpolators ---
n_rows = 100
n_cols = 100
material_law = run_config["problem"]["material_law"]
L_max = int(run_config["problem"]["x_max"])  # in mm
theta = run_config["problem"].get("theta", 0.0)  # in degrees, if applicable
FEM_dataset = f"{material_law}_{n_rows}x{n_cols}{f'_{theta}deg' if material_law == 'orthotropic' else ''}.dat"
fem_file = os.path.join("../data_fem", FEM_dataset)

data = np.loadtxt(fem_file)
X_val = data[:, :2]
u_val = data[:, 2:4]
strain_val = data[:, 4:7]
stress_val = data[:, 7:10]
solution_val = np.hstack((u_val, strain_val, stress_val))
n_mesh_points = int(np.sqrt(X_val.shape[0]))
x_grid = np.linspace(0, L_max, n_mesh_points)
y_grid = np.linspace(0, L_max, n_mesh_points)
interpolators = [RegularGridInterpolator((x_grid, y_grid),
                                          solution_val[:, i].reshape(n_mesh_points, n_mesh_points).T)
                 for i in range(solution_val.shape[1])]
func = lambda x: np.array([interp((x[:, 0], x[:, 1])) for interp in interpolators]).T

# --- Interpolate pde weights to grid ---
X_pde_grid = np.linspace(0, L_max, int(np.sqrt(pde_weights.shape[1])))
Y_pde_grid = np.linspace(0, L_max, int(np.sqrt(pde_weights.shape[1])))
pde_weights_grid = pde_weights.reshape((pde_weights.shape[0], int(np.sqrt(pde_weights.shape[1])), int(np.sqrt(pde_weights.shape[1]))))

pde_weights_interpolated = np.array([RegularGridInterpolator(
    (X_pde_grid, Y_pde_grid),
    pde_weights_grid[it, :, :],
    method='linear', bounds_error=False, fill_value=None
)(X_val) for it in range(pde_weights_grid.shape[0])])

mat_weights_grid = mat_weights.reshape((mat_weights.shape[0], int(np.sqrt(mat_weights.shape[1])), int(np.sqrt(mat_weights.shape[1]))))
mat_weights_interpolated = np.array([RegularGridInterpolator(
    (X_pde_grid, Y_pde_grid),
    mat_weights_grid[it, :, :],
    method='linear', bounds_error=False, fill_value=None
)(X_val) for it in range(mat_weights_grid.shape[0])])

# --- Prepare a square grid for field plotting ---

nx=60
ny=88
Xp = np.loadtxt(f"../deep_notched_{nx}x{ny}.txt")

# Interpolate mapping
X_map_points = Xp[:, 0].reshape((ny, nx)).T
Y_map_points = Xp[:, 1].reshape((ny, nx)).T

def coordMap(x, X_map = X_map_points, Y_map = Y_map_points, x_max = L_max, y_max=L_max, padding=1e-6):
    x_pos = x[0] / x_max * (X_map.shape[0]-1) * (1-2*padding) + padding
    y_pos = x[1] / y_max * (Y_map.shape[1]-1) * (1-2*padding) + padding
    xm = jax.scipy.ndimage.map_coordinates(X_map,
           [x_pos, y_pos], order=1, mode='nearest')
    ym = jax.scipy.ndimage.map_coordinates(Y_map,
           [x_pos, y_pos], order=1, mode='nearest')
    return jnp.stack([xm, ym])

ngrid = 100

# Edge coordinates (for pcolor plotting)
x_edge = np.linspace(0, L_max, ngrid+1)
y_edge = np.linspace(0, L_max, ngrid+1)
Xedge, Yedge = np.meshgrid(x_edge, y_edge, indexing="ij")
Xgrid = jax.vmap(coordMap)(np.stack((Xedge.ravel(), Yedge.ravel()), axis=1))
Xmesh, Ymesh = Xgrid[:,0].reshape(Xedge.shape), Xgrid[:,1].reshape(Yedge.shape)

# Center coordinates (for evaluating fields)
x_center = 0.5 * (x_edge[:-1] + x_edge[1:])
y_center = 0.5 * (y_edge[:-1] + y_edge[1:])
# Xgrid, Ygrid = jax.vmap(coordMap)(np.stack((Xc.ravel(), Yc.ravel()), axis=1))
Xgrid, Ygrid = np.meshgrid(x_center, y_center, indexing="ij")
# Xgrid, Ygrid = jax.vmap(coordMap)(np.stack((Xc.ravel(), Yc.ravel()), axis=1))
# Xgrid = Xgrid[:, 0].reshape(Xc.shape)
# Ygrid = Xgrid[:, 1].reshape(Yc.shape)

# x_edge = np.linspace(0,L_max,ngrid)
# y_edge = np.linspace(0,L_max,ngrid)

# Xgrid, Ygrid = np.meshgrid(x_edge, y_edge, indexing="ij")
# Xmesh = jax.vmap(coordMap)(np.stack((Xgrid.ravel(), Ygrid.ravel()), axis=1))
# Xmesh, Ymesh = Xmesh[:,0].reshape(Xgrid.shape), Xmesh[:,1].reshape(Ygrid.shape)

# Xmesh, Ymesh = np.meshgrid(np.linspace(0, L_max, ngrid), np.linspace(0, L_max, ngrid), indexing='ij')
# normdiff = set_normdiff(iteration, fields, fields_id, func, Xmesh, Ymesh)

# --- Plotting ---
iteration = int(len(steps)*1) - 1

subplot_width = plotting_factor
fig_fields, ax = plt.subplots(3,plot_metrics+n_fields,figsize = (plot_metrics*subplot_width+n_fields*subplot_width,subplot_width*3),dpi = 40*plot_metrics+40*n_fields)
printed_font_size = 6
title_font_size = plt.rcParams['font.size']
axes_font_size = ax[0][0].get_xticklabels()[0].get_fontsize()
print(f"Fig width: {fig_fields.get_figwidth()}, set it to {fig_fields.get_figwidth()*printed_font_size/title_font_size:.2f} < 6.34 (A4 with margins) in latex to have a printed font size of {printed_font_size} for titles and {axes_font_size*printed_font_size/title_font_size:.2f} for the axis")


# Plot mat weights on ax_mat_weights
if run_config["execution"]["Self-Attention"]:# and not run_config["execution"]["SA_share_weights"]:

    # Plot pde weights on ax_pde_weights
    ax_pde_weights = ax[0][0]
    im_pde_weights = pcolor_plot(ax_pde_weights, Xmesh, Ymesh, pde_weights_interpolated[iteration, :].reshape(Xgrid.shape),
                                    r"$\lambda_{PDE}$", colormap='Blues')
    pos = ax_pde_weights.get_position()
    cax_pos = mtransforms.Bbox.from_bounds(pos.x0 - 0.015, pos.y0 + 0.05*pos.height,
                                            0.005, pos.height*0.9)
    cax = fig_fields.add_axes(cax_pos)
    cb_pde_weights = fig_fields.colorbar(im_pde_weights, cax=cax, orientation='vertical', format=make_formatter())
    cb_pde_weights.ax.yaxis.set_ticks_position('left')

    ims_weights = [im_pde_weights]
    fields_weights = [ pde_weights_interpolated] 
    cb_weights = [cb_pde_weights]

    if not run_config["execution"]["SA_share_weights"]: 
        ax_mat_weights = ax[1][0]
        im_mat_weights = pcolor_plot(ax_mat_weights, Xmesh, Ymesh, mat_weights_interpolated[iteration, :].reshape(Xgrid.shape),
                                        r"$\lambda_{Mat}$", colormap='Blues')
        pos = ax_mat_weights.get_position()
        cax_pos = mtransforms.Bbox.from_bounds(pos.x0 - 0.015, pos.y0 + 0.05*pos.height,
                                                0.005, pos.height*0.9)
        cax = fig_fields.add_axes(cax_pos)
        cb_mat_weights = fig_fields.colorbar(im_mat_weights, cax=cax, orientation='vertical', format=make_formatter())
        cb_mat_weights.ax.yaxis.set_ticks_position('left')

        ims_weights.append(im_mat_weights)
        fields_weights.append(mat_weights_interpolated)
        cb_weights.append(cb_mat_weights)

else :
    ims_weights = []
    fields_weights = []
    cb_weights = []

ax[0][0].set_axis_off()
ax[1][0].set_axis_off()

if plot_metrics:  
    # Plot metrics on ax_metric
    lines, scatters = init_metrics(ax[2][0], steps, metrics, metrics_names, 
                                step_type=step_type, time_unit=time_unit, metrics_idx=[0])
    update_metrics(iteration, lines, scatters, steps, metrics, metrics_idx=[0])

ims_field = []
ims_ref = []
ims_res = []
ax_fields = ax[:, plot_metrics:]
ims_ref, ims_field, ims_res = plot_field_columns(fig_fields, ax_fields, fields, fields_plotted, iteration,
                                                  Xmesh, Ymesh, Xgrid, Ygrid, func, field_names_tex, cmap, make_formatter)
